In [1]:
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [2]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Create empty graph
G = nx.Graph()

# Load cypedge edgelist as an unweighted graph
path = os.path.join("..", "Datasets", "carrib.txt") 

# Load columns 0 and 1 as strings to handle the 'V' prefix safely
edges_str = np.loadtxt(path, dtype=str, usecols=(0, 1))

# Strip the 'V' prefix from each node label and convert to an integer
edges = np.array([[int(node.replace('V', '')) for node in row] for row in edges_str])

# If the file has a single edge, np.loadtxt returns a 1D array, so normalize it.
if edges.ndim == 1:
    edges = edges.reshape(1, 2)

G.add_edges_from(edges)

# Basic info
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Draw graph only if it is reasonably small; otherwise skip plotting.
if G.number_of_nodes() <= 200:
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=50, font_size=8)
    plt.show()
else:
    print("Graph is large; skipping full plot.")


Nodes: 249
Edges: 3503
Graph is large; skipping full plot.


In [3]:
# adjacency matrix
nodelist = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodelist)
print(A)

[[0. 1. 1. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]]


In [4]:
#Degree
deg = np.array([G.degree(n) for n in nodelist])
deg

array([ 71,  25,  37, 105,  97,   6,  10,  59,  22, 126,  13, 127,  26,
        50,  17,  16,  25,  23,  22,  22,   9,  21,  11,  19,  13,  17,
        24,  14,  12,   8,  17,   5,   5,  34,   8,  14,  11,  33,  28,
        14,  13,  17,  19,  13,  19,   3,  11,   9,  13,  13,  20,  17,
        21,  30,  25,  29,  25,  30,  22,  17,  21,  21,  27,  18,  24,
        18,  19,  10,  14,  21,  27, 248,  31,  21,  19,  22,  17,  16,
        10,  13,  26,  11,  68,  13,  46,  10,  43,  23,  25,  23, 106,
        21,  26,  57,   3,  29,  27,  25,  19,  17,  17,  27,  16,  20,
        17,  43,  12,  18,  13,  15,  17,  17,  23,  36,  22,  16,  16,
        20,  10,  10,  36,  35,  32,  28,  17,  18,  25,  25,  23,  15,
        25,  15,  23,  53,  17,  28,  31,  11,  14,  13,  14,  16,  14,
        21,  20,  16,  16,  19,  20,  21,  14,  53,  30,  30,  23,  24,
        21,  40,  25,  25,  19,  20, 112,  14,  16,   3,  36,   8, 114,
        17,  22,  16, 116,  46,  13,  14,  15, 170, 108,  16,  1

In [5]:
dist = dict(nx.all_pairs_shortest_path_length(G))
dist

{np.int64(1): {np.int64(1): 0,
  np.int64(12): 1,
  np.int64(14): 1,
  np.int64(15): 1,
  np.int64(20): 1,
  np.int64(21): 1,
  np.int64(22): 1,
  np.int64(25): 1,
  np.int64(26): 1,
  np.int64(28): 1,
  np.int64(29): 1,
  np.int64(32): 1,
  np.int64(34): 1,
  np.int64(36): 1,
  np.int64(37): 1,
  np.int64(39): 1,
  np.int64(41): 1,
  np.int64(42): 1,
  np.int64(43): 1,
  np.int64(51): 1,
  np.int64(52): 1,
  np.int64(53): 1,
  np.int64(58): 1,
  np.int64(59): 1,
  np.int64(60): 1,
  np.int64(61): 1,
  np.int64(90): 1,
  np.int64(102): 1,
  np.int64(107): 1,
  np.int64(108): 1,
  np.int64(125): 1,
  np.int64(132): 1,
  np.int64(133): 1,
  np.int64(151): 1,
  np.int64(155): 1,
  np.int64(156): 1,
  np.int64(157): 1,
  np.int64(158): 1,
  np.int64(159): 1,
  np.int64(160): 1,
  np.int64(167): 1,
  np.int64(175): 1,
  np.int64(176): 1,
  np.int64(177): 1,
  np.int64(178): 1,
  np.int64(180): 1,
  np.int64(181): 1,
  np.int64(182): 1,
  np.int64(183): 1,
  np.int64(184): 1,
  np.int64(185)

In [6]:
n = len(nodelist)
dist_matrix = np.zeros((n, n))

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and v in dist[u]:
            dist_matrix[i, j] = dist[u][v]

print(dist_matrix)

[[0. 1. 1. ... 2. 2. 2.]
 [1. 0. 2. ... 2. 2. 2.]
 [1. 2. 0. ... 2. 2. 2.]
 ...
 [2. 2. 2. ... 0. 2. 1.]
 [2. 2. 2. ... 2. 0. 2.]
 [2. 2. 2. ... 1. 2. 0.]]


4. Node Feature Extraction (Exact Formulas)

Paper defines three measures.

NLI  = Local influence

NGI  = Global influence

NLGC = Hybrid influence

### 4.1 Global Influence (NGI)

**Paper formula:**

$$\text{NGI}_i = \sum_{i \neq j} \frac{\sqrt{d(v_j) + \alpha}}{d_{ij}}$$

**Where:**
*   $d(v_j)$ = degree of node $j$
*   $d_{ij}$ = shortest path distance between node $i$ and node $j$
*   $\alpha$ = constant parameter

### 🔹 Node Global Influence (NGI) – Description

**Definition:**
NGI is a metric used to measure the overall importance of a node by considering its interaction with all other nodes in the network.

**Core Idea:**
It combines both:
*   **Local information** → node degree
*   **Global information** → shortest path distance

**Computation:**
For each node, influence is calculated by summing contributions from all other nodes based on:
*   **Smoothed degree** of the contributing node (using square root scaling)
*   **Distance** between the two nodes

**Role of Degree ($d(v_j)$):**
Nodes with higher connections contribute more influence. However, instead of using the raw degree directly, a square root transformation is applied to moderate the dominance of high-degree nodes.

**Role of Distance ($d_{ij}$):**
Influence decreases as distance increases, ensuring closer nodes have stronger impact. The inverse relationship gives higher weight to nearby nodes.

**Role of $\alpha$ (alpha):**
*   Added inside the square root to stabilize the computation.
*   Prevents zero or very small degree values from reducing influence too much.
*   Helps in smoothing the contribution of nodes.
*   $\alpha = 0.5$ provides a balanced contribution.

**Key Advantage:**
NGI captures both local connectivity and global positioning while ensuring balanced influence using square root scaling.

**Interpretation:**
A node with a higher NGI value is more influential in the network, considering both its connectivity and its position relative to other nodes.

In [7]:
alpha = 0.5

NGI = np.zeros(n)

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and dist_matrix[i, j] != 0:
            NGI[i] += np.sqrt(deg[j] + alpha) / dist_matrix[i, j]

print("NGI:", NGI)

NGI: [ 781.15521036  689.16648456  717.83264076  892.72960584  855.06904071
  639.99637477  642.42216368  771.60674284  682.66290394  942.56642158
  666.64536803  941.15058525  690.21150196  737.56851172  664.07808915
  670.82349347  718.13095056  716.0482926   707.69900597  702.25639194
  658.52508036  713.42772009  654.33178819  694.06495523  659.31490104
  687.83847519  717.72428243  671.95288252  659.33761347  648.50480406
  695.75879429  644.17307698  644.17307698  731.72586638  656.32223532
  672.92626822  659.72476393  746.5613565   736.75015734  685.82970382
  672.49995589  687.1836571   698.04660018  673.16741419  696.98971775
  631.58664499  657.62401074  654.51000898  665.87638641  665.87638641
  709.44369929  699.22392107  715.32885181  748.99825526  724.84692141
  735.76718051  721.61244852  730.59475494  710.04696773  684.11861665
  702.20807711  694.47362865  722.02946937  686.2845873   717.56546242
  683.64916927  686.39805341  666.3042059   688.98367934  700.47816776
 

### 4.2 Node Local Influence (NLI)

**Formula:**

$$NLI_i = \frac{d(v_i) \times \log_2 \left( \sum_{j \in N_i} e^{d(v_j)} \right)}{n}$$

---

### 🔹 Node Local Influence (NLI) – Description

**Definition:**
NLI is a metric used to measure the importance of a node based on its local neighborhood structure, focusing only on its immediate connections.

**Core Idea:**
It captures influence using:
*   **Node’s own degree**
*   **Contribution from its neighboring nodes**

**Computation:**
For each node, influence is calculated by:
1. Taking the degree of the node.
2. Multiplying it with the logarithm of the sum of exponential contributions from its neighbors.
3. Normalizing by the total number of nodes ($n$).

**Role of Degree ($d(v_i)$):**
The degree reflects direct connectivity; nodes with more neighbors have higher local influence.

**Role of Neighbor Contribution:**
Each neighbor contributes via an exponential function, which:
*   Amplifies the importance of highly connected neighbors.
*   Highlights strong local structures.

**Role of Logarithm ($\\log$):**
*   Compresses large values from exponential growth.
*   Prevents numerical explosion and ensures balanced scaling.

**Role of Normalization ($n$):**
Dividing by total nodes ensures values are comparable across different graph sizes.

**Key Advantage:**
NLI focuses purely on local structure, making it effective in identifying nodes that are well-connected locally and surrounded by influential neighbors.

**Interpretation:**
A node with higher NLI is more influential within its immediate neighborhood, even if it is not globally central.

In [8]:
NLI = np.zeros(n)

for i, u in enumerate(nodelist):

    neighbors = list(G.neighbors(u))

    # Sum of exponential terms (using neighbor count here as influence proxy)
    exp_sum = 0
    for v in neighbors:
        exp_sum += np.exp(G.degree(v))   # you can modify this part if Ne_i(v_i) defined differently

    if exp_sum > 0:
        NLI[i] = (G.degree(u) * np.log2(exp_sum)) / n
    else:
        NLI[i] = 0

print("NLI:", NLI)

NLI: [102.01997703  35.92252712  53.16534014 150.87461391 139.37940524
   8.62140651  14.36901085  84.77716401  31.61182387 181.0495367
  18.6797141  182.48643778  37.35942821  71.84505425  24.42731844
  22.99041736  35.92252712  33.04872495  31.61182387  31.61182387
  12.93210976  30.17492278  15.80591193  27.30112061  18.6797141
  24.42731844  34.48562604  20.11661519  17.24281302  11.49520868
  24.42731844   7.18450542   7.18450542  48.85463689  11.49520868
  20.11661519  15.80591193  47.4177358   40.23323038  20.11661519
  18.6797141   24.42731844  27.30112061  18.6797141   27.30112061
   4.31070325  15.80591193  12.93210976  18.6797141   18.6797141
  28.7380217   24.42731844  30.17492278  43.10703255  35.92252712
  41.67013146  35.92252712  43.10703255  31.61182387  24.42731844
  30.17492278  30.17492278  38.79632929  25.86421953  34.48562604
  25.86421953  27.30112061  14.36901085  20.11661519  30.17492278
  38.79632929 288.81711807  44.54393363  30.17492278  27.30112061
  31.611

4.3 Hybrid Influence

Paper multiplies them.

$$\text{NLGC}_i = \text{NLI}_i \times \text{NGI}_i$$

In [9]:
NLGC = NLI * NGI
NLGC

array([ 79693.43661643,  24756.6017337 ,  38163.81651048, 134690.23461097,
       119179.01432931,   5517.66891146,   9230.97103963,  65414.63138835,
        21580.21948039, 170651.21393371,  12452.74488333, 171747.21771991,
        25785.90705559,  52990.64973389,  16221.64695502,  15422.51208879,
        25797.07854887,  23664.483075  ,  22371.65632806,  22199.60537197,
         8516.11862159,  21527.62636507,  10342.31061963,  18948.75105614,
        12315.81385572,  16802.04947092,  24751.17120214,  13517.41756262,
        11368.83518536,   7454.69805217,  16995.52162777,   4628.06496592,
         4628.06496592,  35748.20150248,   7544.56105578,  13536.99878805,
        10427.55151925,  35400.24916229,  29641.83881087,  13796.57223671,
        12562.10691082,  16786.05402114,  19057.45442514,  12574.57484109,
        19028.60035051,   2722.5826062 ,  10394.34719939,   8464.19527784,
        12438.38052656,  12438.38052656,  20388.00842368,  17080.16538317,
        21584.99286773,  

### 🔹 Multi-Scale Feature Construction

**Definition:**
Multi-scale feature construction is used to capture node influence at different neighborhood levels by progressively aggregating information from neighboring nodes.

**Core Idea:**
Instead of relying on a single-scale measure, influence is computed across multiple levels:
*   **Level 1** → node itself
*   **Level 2** → node + immediate neighbors
*   **Level 3** → node + extended neighborhood

---

### 🧬 Computation: NLI-based Features

**Level 1:**
$$W_{NLI1}(i) = NLI_i$$

**Level 2:**
$$W_{NLI2}(i) = W_{NLI1}(i) + \sum_{j \in N(i)} W_{NLI1}(j)$$

**Level 3:**
$$W_{NLI3}(i) = W_{NLI2}(i) + \sum_{j \in N(i)} W_{NLI2}(j)$$

---

### 🌍 Computation: NGI-based Features

**Level 1:**
$$W_{NGI1}(i) = NGI_i$$

**Level 2:**
$$W_{NGI2}(i) = W_{NGI1}(i) + \sum_{j \in N(i)} W_{NGI1}(j)$$

**Level 3:**
$$W_{NGI3}(i) = W_{NGI2}(i) + \sum_{j \in N(i)} W_{NGI2}(j)$$

---

### ✅ Key Advantages
*   **Higher-Order Influence:** Captures both local and extended neighborhood importance.
*   **Rich Structural Info:** Provides a multidimensional view of a node's position for learning.
*   **Propagation Awareness:** Helps GCNs understand how influence spreads across multiple hops.

**Interpretation:**
Nodes with higher values at deeper levels (e.g., $NLI_3$, $NGI_3$) are not only locally important but are also strategically connected to other influential regions in the graph.

In [10]:
import numpy as np

nodelist = list(G.nodes())
node_index = {node: i for i, node in enumerate(nodelist)}
n = len(nodelist)

# --- NLI Multi-scale ---
W_NLI1 = NLI.copy()
W_NLI2 = np.zeros(n)
W_NLI3 = np.zeros(n)

# NLI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI1[j]
    W_NLI2[i] = W_NLI1[i] + neighbor_sum

# NLI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI2[j]
    W_NLI3[i] = W_NLI2[i] + neighbor_sum


# --- NGI Multi-scale ---
W_NGI1 = NGI.copy()
W_NGI2 = np.zeros(n)
W_NGI3 = np.zeros(n)

# NGI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI1[j]
    W_NGI2[i] = W_NGI1[i] + neighbor_sum

# NGI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI2[j]
    W_NGI3[i] = W_NGI2[i] + neighbor_sum


print("W_NLI1:", W_NLI1)
print("W_NLI2:", W_NLI2)
print("W_NLI3:", W_NLI3)

print("W_NGI1:", W_NGI1)
print("W_NGI2:", W_NGI2)
print("W_NGI3:", W_NGI3)

W_NLI1: [102.01997703  35.92252712  53.16534014 150.87461391 139.37940524
   8.62140651  14.36901085  84.77716401  31.61182387 181.0495367
  18.6797141  182.48643778  37.35942821  71.84505425  24.42731844
  22.99041736  35.92252712  33.04872495  31.61182387  31.61182387
  12.93210976  30.17492278  15.80591193  27.30112061  18.6797141
  24.42731844  34.48562604  20.11661519  17.24281302  11.49520868
  24.42731844   7.18450542   7.18450542  48.85463689  11.49520868
  20.11661519  15.80591193  47.4177358   40.23323038  20.11661519
  18.6797141   24.42731844  27.30112061  18.6797141   27.30112061
   4.31070325  15.80591193  12.93210976  18.6797141   18.6797141
  28.7380217   24.42731844  30.17492278  43.10703255  35.92252712
  41.67013146  35.92252712  43.10703255  31.61182387  24.42731844
  30.17492278  30.17492278  38.79632929  25.86421953  34.48562604
  25.86421953  27.30112061  14.36901085  20.11661519  30.17492278
  38.79632929 288.81711807  44.54393363  30.17492278  27.30112061
  31.

### 🔹 Neighborhood Matrix Construction

**Definition:**
A neighborhood matrix is constructed for each node to represent its local structural information using a fixed-size subgraph.

**Core Idea:**
Instead of using the entire graph, a localized neighborhood subgraph is extracted for each node, ensuring:
*   Reduced computational complexity
*   Consistent input size for learning models

---

### ⚙️ Computation Steps:
1.  **Extract** one-hop neighbors of the target node.
2.  **Rank** neighbors based on importance scores (e.g., $W_{NLI3}$ or $W_{NGI3}$).
3.  **Select** the top $L$ neighbors.
4.  **Construct** a $(L+1) 	imes (L+1)$ adjacency matrix including the node and selected neighbors.

**Role of Parameter $L$:**
*   Determines the size of the neighborhood.
*   Controls how much local information is captured.
*   Ensures uniform matrix size across all nodes.

---

### ✅ Key Advantage
*   **Efficiency:** Reduces computational complexity.
*   **Robustness:** Avoids bias from high-degree nodes.
*   **Consistency:** Provides structured and consistent input for GCN.

**Interpretation:**
Each node is represented by a fixed-size local subgraph, capturing its most important neighbors and their mutual connections.

In [11]:
import numpy as np

# choose L <= max neighbors: use a fixed neighborhood size of 40 for the large graph
L = 40

def neighborhood_matrix(node):

    nbrs = list(G.neighbors(node))

    # sort neighbors using importance (W_NLI3 or W_NGI3)
    nbrs_sorted = sorted(nbrs, key=lambda x: W_NLI3[nodelist.index(x)], reverse=True)

    nbrs_selected = nbrs_sorted[:L]

    # keep a fixed size L+1; pad with placeholder values if the node has fewer neighbors
    nodes = [node] + nbrs_selected
    if len(nodes) < L + 1:
        nodes += [None] * (L + 1 - len(nodes))

    size = L + 1
    mat = np.zeros((size, size))

    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if u is not None and v is not None and G.has_edge(u, v):
                mat[i, j] = 1

    return mat, nodes


# Example: for the first node in the graph
mat0, nodes0 = neighborhood_matrix(nodelist[0])

print("Neighborhood Matrix:\n", mat0)
print("Nodes used:", nodes0)

Neighborhood Matrix:
 [[0. 1. 1. ... 1. 1. 1.]
 [1. 0. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 0. 0. 0.]
 ...
 [1. 1. 0. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]]
Nodes used: [np.int64(1), np.int64(249), np.int64(28), np.int64(32), np.int64(15), np.int64(20), np.int64(25), np.int64(36), np.int64(190), np.int64(158), np.int64(159), np.int64(14), np.int64(151), np.int64(192), np.int64(196), np.int64(247), np.int64(41), np.int64(191), np.int64(42), np.int64(201), np.int64(203), np.int64(193), np.int64(90), np.int64(189), np.int64(53), np.int64(197), np.int64(185), np.int64(43), np.int64(12), np.int64(199), np.int64(186), np.int64(51), np.int64(59), np.int64(176), np.int64(26), np.int64(125), np.int64(178), np.int64(246), np.int64(200), np.int64(61), np.int64(238)]


### 🔹 Structural Channel Construction

**Definition:**
Structural channel construction embeds node feature information into the neighborhood matrix to generate multiple feature-aware representations of each node.

**Core Idea:**
Instead of using only structural adjacency, node features are incorporated into the matrix to create channels that capture both:
*   **Structural relationships**
*   **Node importance**

---

### ⚙️ Computation:
For each node, a neighborhood matrix is constructed and node features (e.g., NLI, NGI) are embedded into this matrix according to specific rules:

**Channel Construction Rules:**
*   **Diagonal elements:** Represent the feature value of the node itself.
*   **Off-diagonal elements:**
    *   If an edge exists → assign the feature value of the neighbor.
    *   If no edge exists → the value remains zero.

**Channels Created:**
*   **Local influence channels:** $E^{(NLI1)}$, $E^{(NLI2)}$, $E^{(NLI3)}$
*   **Global influence channels:** $E^{(NGI1)}$, $E^{(NGI2)}$, $E^{(NGI3)}$

---

### ✅ Key Advantage
*   **Integration:** Combines structural and feature information seamlessly.
*   **Power:** Enhances the representation power of nodes.
*   **Scalability:** Provides multi-scale learning capability.

**Interpretation:**
Each channel represents a feature-enriched local subgraph, enabling the model to learn both node importance and connectivity patterns simultaneously.

In [12]:
import numpy as np


def embed_channel(mat, nodes, feature_dict):

    size = mat.shape[0]
    out = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            u = nodes[i]
            v = nodes[j]

            # diagonal → self feature
            if i == j:
                out[i, j] = feature_dict.get(u, 0)

            # edge exists → take neighbor feature
            elif mat[i, j] == 1:
                out[i, j] = feature_dict.get(v, 0)

    return out

### 🔹 Purpose of Structural Channel Construction

Structural channel construction is performed to transform the graph into a format that can effectively capture both **node importance** and **local structural relationships** in a unified representation.

Graph data is inherently irregular, where each node may have a different number of neighbors. This makes it difficult to directly apply deep learning models that require fixed-size inputs.

To address this, a neighborhood matrix is first constructed for each node, ensuring a consistent structure. However, this matrix only represents connectivity and does not include any information about node importance.

Therefore, node features such as local influence (NLI) and global influence (NGI) are embedded into the neighborhood matrix to create **feature-aware channels**.

**In these channels:**
*   The **diagonal elements** represent the importance of the node itself
*   The **off-diagonal elements** represent the importance of neighboring nodes if a connection exists

This transformation allows the model to simultaneously learn:
1.  **Who is connected to whom** (structure)
2.  **How important each node is** (features)

By constructing multiple channels at different scales (NLI1–3 and NGI1–3), the model is able to capture multi-level influence propagation, improving its ability to identify key nodes.

---

### ✅ Key Benefit
This approach enables the graph to be represented as a **multi-channel matrix** (similar to images), making it suitable for deep learning models while preserving both structural and semantic information.

In [13]:
# convert arrays to dict (important)
NLI_dict = {node: NLI[i] for i, node in enumerate(nodelist)}
NGI_dict = {node: NGI[i] for i, node in enumerate(nodelist)}

# example for node 3
mat, nodes = neighborhood_matrix(3)

E_NLI1 = embed_channel(mat, nodes, NLI_dict)
E_NLI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI2)))
E_NLI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI3)))

E_NGI1 = embed_channel(mat, nodes, NGI_dict)
E_NGI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI2)))
E_NGI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI3)))

print("E_NLI1:\n", E_NLI1)
print("E_NLI2:\n", E_NLI2)
print("E_NLI3:\n", E_NLI3)
print("E_NGI1:\n", E_NGI1)
print("E_NGI2:\n", E_NGI2)
print("E_NGI3:\n", E_NGI3)

E_NLI1:
 [[ 18.6797141  288.81711807 182.48643778 ...   0.           0.
    0.        ]
 [ 18.6797141  288.81711807 182.48643778 ...   0.           0.
    0.        ]
 [ 18.6797141  288.81711807 182.48643778 ...   0.           0.
    0.        ]
 ...
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]]
E_NLI2:
 [[1043.19018764 9999.39464983 6239.02451064 ...    0.
     0.            0.        ]
 [1043.19018764 9999.39464983 6239.02451064 ...    0.
     0.            0.        ]
 [1043.19018764 9999.39464983 6239.02451064 ...    0.
     0.            0.        ]
 ...
 [   0.            0.            0.         ...    0.
     0.            0.        ]
 [   0.            0.            0.         ...    0.
     0.            0.        ]
 [   0.            0.            0.         ...    0.
     0.           

### 🔹 Channel Tensor Construction

**Definition:**
Channel tensor construction combines multiple feature-embedded neighborhood matrices into a unified multi-dimensional representation for each node.

**Core Idea:**
For each node, six structural channels are generated by embedding different feature representations ($NLI_1$–$NLI_3$ and $NGI_1$–$NGI_3$) into the neighborhood matrix.

---

### ⚙️ Computation:
1.  A **neighborhood matrix** of size $(L+1) \times (L+1)$ is constructed.
2.  **Six feature matrices** are generated by embedding node importance values into the structural layout.
3.  These matrices are stacked to form a tensor of size:
    $$6 \times (L+1) \times (L+1)$$

---

### ✅ Key Advantage
*   **Multi-Perspective:** Captures multiple levels of node importance.
*   **Hybrid Representation:** Combines structural connectivity and feature importance.
*   **Deep Learning Ready:** Enables standard CNN or GCN models to process graph data efficiently.

**Interpretation:**
Each node is represented as a multi-channel tensor, where each channel encodes a different aspect of node influence and neighborhood structure.

In [14]:
channels = []

for node in G.nodes():

    mat, nodes = neighborhood_matrix(node)

    # create feature dicts (node → value), skipping None placeholders
    f1 = {n: NLI_dict.get(n, 0) for n in nodes}
    f2 = {n: W_NLI2[node_index[n]] if n is not None else 0 for n in nodes}
    f3 = {n: W_NLI3[node_index[n]] if n is not None else 0 for n in nodes}

    f4 = {n: NGI_dict.get(n, 0) for n in nodes}
    f5 = {n: W_NGI2[node_index[n]] if n is not None else 0 for n in nodes}
    f6 = {n: W_NGI3[node_index[n]] if n is not None else 0 for n in nodes}

    # create channels
    c1 = embed_channel(mat, nodes, f1)
    c2 = embed_channel(mat, nodes, f2)
    c3 = embed_channel(mat, nodes, f3)

    c4 = embed_channel(mat, nodes, f4)
    c5 = embed_channel(mat, nodes, f5)
    c6 = embed_channel(mat, nodes, f6)

    # stack → (6, L+1, L+1)
    tensor = np.stack([c1, c2, c3, c4, c5, c6])

    channels.append(tensor)

# final shape: (num_nodes, 6, L+1, L+1)
channels = np.array(channels)

print(channels.shape)

(249, 6, 41, 41)


### 🔹 Channel Attention Module

**Definition:**
The channel attention module is used to adaptively learn the importance of different feature channels and enhance the representation of informative channels.

**Core Idea:**
Not all feature channels contribute equally to node importance. Therefore, an attention mechanism is introduced to assign weights to each channel dynamically.

---

### ⚙️ Computation:
1.  **Global average pooling** is applied to each channel to obtain a compact representation.
2.  The pooled values are passed through **fully connected layers**.
3.  A **sigmoid activation** generates normalized weights between 0 and 1.
4.  These weights are **multiplied** with the input feature maps.

---

### ✅ Key Advantage
*   **Feature Selection:** Highlights important feature channels.
*   **Noise Reduction:** Suppresses less relevant information.
*   **Robustness:** Improves model robustness and generalization.

**Interpretation:**
Channels representing more meaningful structural or influence patterns receive higher weights, allowing the model to focus on the most relevant information.

In [15]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):

    def __init__(self, channels=6, reduction=2):
        super(ChannelAttention, self).__init__()

        # Global Average Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # Fully Connected Layers (SE block)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, channels, height, width)

        b, c, _, _ = x.size()

        # Step 1: Global Average Pooling
        y = self.avg_pool(x).view(b, c)

        # Step 2: FC → channel weights
        y = self.fc(y).view(b, c, 1, 1)

        # Step 3: Multiply weights
        out = x * y

        return out

In [16]:
# test input
x = torch.randn(2, 6, 3, 3)   # batch=2, channels=6

model = ChannelAttention(6)

out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([2, 6, 3, 3])
Output shape: torch.Size([2, 6, 3, 3])


### 🔹 NLGCN Model Architecture

**Definition:**
The NLGCN model is a convolutional neural network designed to learn node influence from multi-channel structural representations of graph data.

**Core Idea:**
The model processes the constructed channel tensor using convolutional layers to extract structural patterns, while a channel attention mechanism enhances important feature channels.

---

### 🏗 Architecture:
*   **Input:** Multi-channel tensor of size $6 \times (L+1) \times (L+1)$.
*   **Channel Attention:** Assigns adaptive weights to feature channels.
*   **Convolution Layer 1:** Extracts local structural patterns (followed by Batch Normalization and ReLU).
*   **Pooling Layer:** Reduces spatial dimensions and retains key features.
*   **Convolution Layer 2:** Learns higher-level structural representations.
*   **Fully Connected Layers:** Transform extracted features into the final influence score.

---

### ✅ Key Advantage
*   **Hybrid Learning:** Captures both local and multi-scale structural patterns.
*   **Attention-Driven:** Enhances feature learning using the attention mechanism.
*   **Structured Processing:** Efficiently processes graph data in a consistent matrix format.

**Interpretation:**
The model learns how node importance is influenced by both its local structure and multi-scale neighborhood features, producing a final influence score.

### 🛠 Model Component Summary

| Part | Role |
| :--- | :--- |
| **Channel Attention** | Adaptively select and weight important feature channels |
| **Convolution Layer 1** | Extract local structural patterns from the neighborhood |
| **Max Pooling** | Reduce spatial dimensions and retain significant features |
| **Convolution Layer 2** | Learn higher-order structural representations |
| **Fully Connected** | Map structural features to the final node influence score |

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLGCN(nn.Module):

    def __init__(self):
        super(NLGCN, self).__init__()

        self.attention = ChannelAttention(6)

        # Conv 1
        self.conv1 = nn.Conv2d(6, 16, kernel_size=2)
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment this block when using larger graphs (L >= 4 or bigger input size)
        # self.conv2 = nn.Conv2d(16, 32, kernel_size=2)
        # self.pool2 = nn.MaxPool2d(2)

        # -------- FC Layers --------
        # For L=40, input size after conv1+pool is 16 x 20 x 20
        self.fc1 = nn.Linear(16 * 20 * 20, 8)
        self.fc2 = nn.Linear(8, 1)

        # For even larger graphs or extra conv layers, adjust this accordingly
        # self.fc1 = nn.Linear(32 * k * k, 64)  # adjust k based on output size
        # self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        # x: (batch, 6, L+1, L+1)

        x = self.attention(x)

        x = self.conv1(x)
        x = self.bn(x)
        x = F.relu(x)

        x = self.pool(x)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment when input size is large enough
        # x = self.conv2(x)
        # x = F.relu(x)
        # x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### 🧬 SIR-Based Label Generation

**Definition:**
The SIR (Susceptible–Infected–Recovered) model is used to generate ground truth labels representing node influence.

---

### ⚙️ Computation:

1.  **Epidemic Threshold:** The threshold is calculated as:
    $$\beta_c = \frac{\langle k \rangle}{\langle k^2 \rangle - \langle k \rangle}$$

2.  **Infection Probability:** The probability is set relative to the threshold:
    $$\beta = 1.5\beta_c$$

3.  **Simulation Process:**
    *   Each node is treated as the initial infected node.
    *   The SIR process is simulated multiple times (e.g., 500 runs).
    *   The average number of recovered nodes is calculated as the influence score.

---

### ✅ Normalization:
The labels are normalized to the range $[0, 1]$ to ensure stable model training.

**Interpretation:**
Nodes that infect a larger portion of the network in the SIR simulation are considered more influential and receive higher ground truth scores.

In [18]:
import numpy as np
import random

# ---- Degree calculations ----
deg = np.array([d for n, d in G.degree()])

k_avg = np.mean(deg)
k2_avg = np.mean(deg**2)

beta_c = k_avg / (k2_avg - k_avg)

beta = 1.5 * beta_c
mu = 1


# ---- SIR Simulation ----
def SIR_simulation(G, seed, beta, mu, steps=1000):

    susceptible = set(G.nodes())
    infected = {seed}
    recovered = set()

    susceptible.remove(seed)

    for _ in range(steps):

        new_infected = set()
        new_recovered = set()

        for node in infected:

            # spread infection
            for nbr in G.neighbors(node):
                if nbr in susceptible:
                    if random.random() < beta:
                        new_infected.add(nbr)

            # recovery
            if random.random() < mu:
                new_recovered.add(node)

        infected |= new_infected
        infected -= new_recovered

        recovered |= new_recovered
        susceptible -= new_infected

        if len(infected) == 0:
            break

    return len(recovered)


# ---- Label Generation ----
labels = []
runs = 500

for node in G.nodes():

    spread = 0

    for _ in range(runs):
        spread += SIR_simulation(G, node, beta, mu)

    labels.append(spread / runs)

labels = np.array(labels)


# ---- Normalize Labels ----
labels = labels / np.max(labels)

print("Labels:", labels)

Labels: [0.45896313 0.24367849 0.32561096 0.7106936  0.58172058 0.10057918
 0.10347507 0.42689645 0.25815793 0.75483825 0.23859302 0.75589773
 0.25851109 0.32222065 0.18350049 0.17227009 0.32448086 0.28033621
 0.3252578  0.26854075 0.16725526 0.28259641 0.14769035 0.28075999
 0.15743749 0.21895748 0.32490465 0.17714366 0.17057494 0.12861986
 0.31388614 0.10658285 0.14302868 0.37046193 0.14585393 0.24629185
 0.14521825 0.4301455  0.36996751 0.24735132 0.21083486 0.22587936
 0.2816782  0.18427744 0.24643311 0.09506993 0.18922164 0.17029241
 0.14345247 0.16923294 0.29615765 0.28782314 0.28746998 0.38642464
 0.29149597 0.35238028 0.27186043 0.3816217  0.32688233 0.22828083
 0.31070773 0.28965956 0.31791213 0.2592174  0.35937279 0.2455149
 0.24509111 0.15934454 0.28881198 0.26041814 0.32335076 1.
 0.23718039 0.19649668 0.2156378  0.19508405 0.25483825 0.27871168
 0.18526628 0.14620709 0.27609832 0.11357536 0.57006639 0.15892075
 0.39490041 0.13886142 0.32271507 0.20850403 0.27496822 0.27080

In [19]:
import torch
import torch.nn as nn

# ---- Convert data to tensors ----
X = torch.tensor(channels, dtype=torch.float32)

# ---- Normalize input channels per channel ----
X = X - X.mean(dim=(0, 2, 3), keepdim=True)
X = X / (X.std(dim=(0, 2, 3), keepdim=True) + 1e-6)

# ---- Normalize labels for stable regression training ----
y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / (y_std + 1e-6)

print("X mean per channel:", X.mean(dim=(0, 2, 3)))
print("X std per channel:", X.std(dim=(0, 2, 3)))
print("y mean:", y_mean.item(), "y std:", y_std.item())

# ---- Initialize model ----
model = NLGCN()

# ---- Optimizer ----
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---- Loss function ----
criterion = nn.MSELoss()

# ---- Training Loop ----
epochs = 300

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

# ---- Final predictions ----
model.eval()
with torch.no_grad():
    predictions = model(X)

print("\nFinal Predictions:\n", predictions)

X mean per channel: tensor([-4.1923e-08,  5.8473e-08,  1.1665e-07, -2.2456e-08, -5.7088e-08,
        -3.6892e-08])
X std per channel: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
y mean: 0.2996784448623657 y std: 0.14778509736061096
Epoch 0, Loss = 1.370828
Epoch 20, Loss = 0.214077
Epoch 40, Loss = 0.143347
Epoch 60, Loss = 0.120692
Epoch 80, Loss = 0.109691
Epoch 100, Loss = 0.100876
Epoch 120, Loss = 0.093551
Epoch 140, Loss = 0.087466
Epoch 160, Loss = 0.082041
Epoch 180, Loss = 0.077100
Epoch 200, Loss = 0.072554
Epoch 220, Loss = 0.068364
Epoch 240, Loss = 0.064313
Epoch 260, Loss = 0.060622
Epoch 280, Loss = 0.057237

Final Predictions:
 tensor([[ 1.0665e+00],
        [-4.2143e-01],
        [ 1.6274e-01],
        [ 2.7478e+00],
        [ 1.8809e+00],
        [-5.1129e-01],
        [-5.1129e-01],
        [ 8.4085e-01],
        [-3.0380e-01],
        [ 3.0516e+00],
        [-5.1129e-01],
        [ 3.0547e+00],
        [-2.8127e-01],
        [ 1.3800e-01],
        [-5.1

### 🔹 Prediction and Ranking Evaluation

**Definition:**
After training, the model predicts influence scores for each node, which are used to rank nodes based on their importance.

---

### ⚙️ Computation:
1.  **Generate Predicted Scores:** The trained model is used to compute influence scores for all nodes in the graph.
2.  **Predicted Ranking:** Nodes are ranked in descending order based on these predicted scores.
3.  **Ground Truth Ranking:** A reference ranking is obtained from the SIR-based simulation labels.
4.  **Comparison:** The predicted ranking is compared with the SIR ranking to measure alignment.

---

### 📊 Evaluation:
*   **Top-k Comparison:** The top-ranked nodes from both predicted and ground truth sets are compared to assess how well the model identifies the most influential nodes.
*   **Ranking Correlation:** Statistical measures can be used to determine the accuracy of the overall node order.

**Key Insight:**
The closer the predicted ranking is to the SIR ranking, the better the model captures the underlying dynamics of node influence within the network.

In [20]:
import numpy as np
import torch

# ---- Prediction ----
model.eval()

with torch.no_grad():
    pred = model(X).detach().cpu().numpy().flatten()

print("Predicted scores:", pred)


# ---- Ranking ----
ranking_pred = np.argsort(pred)[::-1]
ranking_true = np.argsort(labels)[::-1]

print("\nTop predicted nodes:", ranking_pred)
print("Top SIR nodes:", ranking_true)

# Top-k comparison
k = 10
print(f"\nTop {k} predicted nodes:", ranking_pred[:k])
print(f"Top {k} SIR nodes:", ranking_true[:k])

Predicted scores: [ 1.06646919e+00 -4.21427846e-01  1.62742972e-01  2.74776936e+00
  1.88086009e+00 -5.11286080e-01 -5.11286080e-01  8.40850532e-01
 -3.03799450e-01  3.05155444e+00 -5.11286080e-01  3.05471611e+00
 -2.81273425e-01  1.38000429e-01 -5.11286080e-01 -5.11286080e-01
  1.74305797e-01 -1.05493188e-01  1.33515358e-01 -2.06055582e-01
 -5.11286080e-01 -1.01618856e-01 -5.11286080e-01 -1.08910412e-01
 -5.11286080e-01 -4.90581214e-01  1.68386996e-01 -5.11286080e-01
 -5.11286080e-01 -5.11286080e-01 -1.08667225e-01 -5.11286080e-01
 -5.11286080e-01  4.62492645e-01 -5.11286080e-01 -3.73318315e-01
 -5.11286080e-01  8.73866737e-01  4.58576441e-01 -2.95012087e-01
 -5.11286080e-01 -5.11286080e-01 -1.77099168e-01 -5.11286080e-01
 -3.23336542e-01 -5.11286080e-01 -5.11286080e-01 -5.11286080e-01
 -5.11286080e-01 -5.11286080e-01 -3.49910259e-02 -4.28407490e-02
 -6.24896288e-02  5.95266402e-01 -7.22706318e-02  3.51537824e-01
 -2.07033992e-01  5.54335058e-01  1.67244554e-01 -3.95139724e-01
  2.657

###  Model Evaluation

**Kendall Tau Correlation:**
The Kendall Tau coefficient is used to measure the similarity between the predicted node ranking and the SIR-based ground truth ranking. A higher value indicates better agreement between the two rankings.

**Top-N Influence Spread:**
The top-N nodes predicted by the model are selected, and their spreading capability is evaluated using the SIR model. The total number of infected nodes represents the effectiveness of the selected nodes.

---

### ✅ Key Insight:
*   **Kendall Tau:** Evaluates the overall ranking consistency.
*   **Top-N Spread:** Evaluates the practical influence performance of the predicted top nodes.

In [21]:
from scipy.stats import kendalltau

# ---- Kendall Tau Correlation ----
tau, p = kendalltau(pred, labels)

print("Kendall Tau:", tau)


# ---- Top-N Influence Spread ----
N = 3   # for small graph (you can change)

top_node_indices = ranking_pred[:N]
top_nodes = [nodelist[idx] for idx in top_node_indices]

spread_total = 0

for node in top_nodes:
    spread_total += SIR_simulation(G, node, beta, mu)

print("Top-N node indices:", top_node_indices)
print("Top-N nodes:", top_nodes)
print("Spread ability:", spread_total)

Kendall Tau: 0.900898585379636
Top-N node indices: [ 71 237 177]
Top-N nodes: [np.int64(249), np.int64(78), np.int64(79)]
Spread ability: 118


In [22]:
import networkx as nx
from scipy.stats import kendalltau

# Create a copy of G without self-loops for traditional centrality calculations
G_clean = G.copy()
G_clean.remove_edges_from(nx.selfloop_edges(G_clean))

print('Calculating traditional centrality measures for US airports...')

# Degree Centrality
deg_dict = nx.degree_centrality(G_clean)
deg_cent = np.array([deg_dict[n] for n in nodelist])
tau_deg, _ = kendalltau(pred, deg_cent)
print(f'Kendall Tau (Prediction vs Degree): {tau_deg:.4f}')

# Betweenness Centrality
bet_dict = nx.betweenness_centrality(G_clean)
bet_cent = np.array([bet_dict[n] for n in nodelist])
tau_bet, _ = kendalltau(pred, bet_cent)
print(f'Kendall Tau (Prediction vs Betweenness): {tau_bet:.4f}')

# Closeness Centrality
clos_dict = nx.closeness_centrality(G_clean)
clos_cent = np.array([clos_dict[n] for n in nodelist])
tau_clos, _ = kendalltau(pred, clos_cent)
print(f'Kendall Tau (Prediction vs Closeness): {tau_clos:.4f}')

# PageRank
pr_dict = nx.pagerank(G_clean)
pr_cent = np.array([pr_dict[n] for n in nodelist])
tau_pr, _ = kendalltau(pred, pr_cent)
print(f'Kendall Tau (Prediction vs PageRank): {tau_pr:.4f}')

# Coreness (k-core)
core_dict = nx.core_number(G_clean)
core_cent = np.array([core_dict[n] for n in nodelist])
tau_core, _ = kendalltau(pred, core_cent)
print(f'Kendall Tau (Prediction vs Coreness): {tau_core:.4f}')

# Eigenvector Centrality
try:
    eig_dict = nx.eigenvector_centrality(G_clean, max_iter=1000)
    eig_cent = np.array([eig_dict[n] for n in nodelist])
    tau_eig, _ = kendalltau(pred, eig_cent)
    print(f'Kendall Tau (Prediction vs Eigenvector): {tau_eig:.4f}')
except Exception as e:
    print(f'Eigenvector centrality failed: {e}')


Calculating traditional centrality measures for US airports...
Kendall Tau (Prediction vs Degree): 0.7009
Kendall Tau (Prediction vs Betweenness): 0.5001
Kendall Tau (Prediction vs Closeness): 0.7009
Kendall Tau (Prediction vs PageRank): 0.6579
Kendall Tau (Prediction vs Coreness): 0.7366
Kendall Tau (Prediction vs Eigenvector): 0.8287


In [23]:
# ============================================================
# Weighted Centrality Correlation Analysis
# Weight Semantics: HIGH weight = STRONGER connection (positive)
# So weight is directly proportional to influence strength.
# No inversion needed — high weight = more influence flow.
# ============================================================

import os
import numpy as np
import networkx as nx
from scipy.stats import kendalltau

path = os.path.join("..", "Datasets", "carrib.txt")

# Load columns as strings to handle the 'V' prefix safely
raw_str = np.loadtxt(path, dtype=str)

if raw_str.ndim == 1:
    raw_str = raw_str.reshape(1, -1)

# Build a weighted graph
G_weighted = nx.Graph()
for row in raw_str:
    # Strip 'V' prefix and parse node IDs as integers, and weight as float
    u = int(row[0].replace('V', ''))
    v = int(row[1].replace('V', ''))
    w = float(row[2])

    # Safety: avoid zero or negative weights
    w = abs(w) if w != 0 else 1e-6

    # ✅ Positive effect: use weight directly — high weight = strong friendship/influence
    direct_w = w

    if G_weighted.has_edge(u, v):
        existing = G_weighted[u][v]['weight']
        G_weighted[u][v]['weight'] = max(existing, direct_w)  # ✅ keep STRONGEST link
    else:
        G_weighted.add_edge(u, v, weight=direct_w)

G_weighted.remove_edges_from(nx.selfloop_edges(G_weighted))

print(f"Weighted graph — Nodes: {G_weighted.number_of_nodes()}, Edges: {G_weighted.number_of_edges()}")
print(f"Sample weights: {[round(G_weighted[u][v]['weight'], 4) for u,v in list(G_weighted.edges())[:5]]}")

# ---- Weighted Centrality Measures ----
print("\nCalculating weighted centrality measures (positive weight semantics)...")

# Weighted Degree (Strength) — high weight friends boost your score
strength_dict = dict(G_weighted.degree(weight='weight'))
strength_cent = np.array([strength_dict.get(n, 0.0) for n in nodelist])
strength_cent = strength_cent / strength_cent.max() if strength_cent.max() > 0 else strength_cent
tau_wdeg, _ = kendalltau(pred, strength_cent)
print(f"Kendall Tau (Prediction vs Weighted Degree / Strength): {tau_wdeg:.4f}")

# Weighted Betweenness — ✅ distance = 1/weight so high-weight edges are SHORT (preferred paths)
# Strong friendship = easy path = shorter distance
G_weighted_dist = nx.Graph()
for u, v, d in G_weighted.edges(data=True):
    G_weighted_dist.add_edge(u, v, distance=1.0 / d['weight'])  # flip only for path length

wbet_dict = nx.betweenness_centrality(G_weighted_dist, weight='distance', normalized=True)
wbet_cent = np.array([wbet_dict[n] for n in nodelist])
tau_wbet, _ = kendalltau(pred, wbet_cent)
print(f"Kendall Tau (Prediction vs Weighted Betweenness):       {tau_wbet:.4f}")

# Weighted Closeness — same distance trick: strong links = short distance
wclos_dict = nx.closeness_centrality(G_weighted_dist, distance='distance')
wclos_cent = np.array([wclos_dict[n] for n in nodelist])
tau_wclos, _ = kendalltau(pred, wclos_cent)
print(f"Kendall Tau (Prediction vs Weighted Closeness):         {tau_wclos:.4f}")

# Weighted PageRank — high weight = more rank flows through that edge (direct use)
wpr_dict = nx.pagerank(G_weighted, weight='weight')
wpr_cent = np.array([wpr_dict[n] for n in nodelist])
tau_wpr, _ = kendalltau(pred, wpr_cent)
print(f"Kendall Tau (Prediction vs Weighted PageRank):          {tau_wpr:.4f}")

# Weighted Eigenvector — high weight neighbors contribute MORE to your score
try:
    weig_dict = nx.eigenvector_centrality(G_weighted, weight='weight', max_iter=1000)
    weig_cent = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")
except nx.PowerIterationFailedConvergence:
    print("Weighted Eigenvector did not converge — trying numpy fallback...")
    weig_dict = nx.eigenvector_centrality_numpy(G_weighted, weight='weight')
    weig_cent = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")

# ---- Comparison Table ----
print("\n" + "="*62)
print(f"{'Measure(carrib)':<30} {'Unweighted':>12} {'Weighted(+)':>12}")
print("="*62)
print(f"{'Degree / Strength':<30} {tau_deg:>12.4f} {tau_wdeg:>12.4f}")
print(f"{'Betweenness':<30} {tau_bet:>12.4f} {tau_wbet:>12.4f}")
print(f"{'Closeness':<30} {tau_clos:>12.4f} {tau_wclos:>12.4f}")
print(f"{'PageRank':<30} {tau_pr:>12.4f} {tau_wpr:>12.4f}")
print(f"{'Eigenvector':<30} {tau_eig:>12.4f} {tau_weig:>12.4f}")
print("="*62)
print("Weight semantics: high raw weight = strong/friendly edge")
print("High-weight edges = more influence flow (used directly)")

Weighted graph — Nodes: 249, Edges: 3492
Sample weights: [0.0, 0.0, 0.0, 0.0, 0.0]

Calculating weighted centrality measures (positive weight semantics)...
Kendall Tau (Prediction vs Weighted Degree / Strength): 0.1503
Kendall Tau (Prediction vs Weighted Betweenness):       0.1584
Kendall Tau (Prediction vs Weighted Closeness):         0.1371
Kendall Tau (Prediction vs Weighted PageRank):          0.1662
Kendall Tau (Prediction vs Weighted Eigenvector):       0.1430

Measure(carrib)                  Unweighted  Weighted(+)
Degree / Strength                    0.7009       0.1503
Betweenness                          0.5001       0.1584
Closeness                            0.7009       0.1371
PageRank                             0.6579       0.1662
Eigenvector                          0.8287       0.1430
Weight semantics: high raw weight = strong/friendly edge
High-weight edges = more influence flow (used directly)
